In [1]:
import pandas as pd

data = {
    "date": ["2023-04-01", "2023-04-01", "2023-04-01", "2023-04-02", "2023-04-02", "2023-04-03", "2023-04-03"],
    "product": ["Apple", "Banana", "Apple", "Banana", "Banana", "Apple", "Banana"],
    "sales": [100, 200, 150, 100, 150, 200, 180]
}

df = pd.DataFrame(data)

df

,date,product,sales
0,2023-04-01,Apple,100
1,2023-04-01,Banana,200
2,2023-04-01,Apple,150
3,2023-04-02,Banana,100
4,2023-04-02,Banana,150
5,2023-04-03,Apple,200
6,2023-04-03,Banana,180


In [2]:
df.groupby(["date", "product"]).cumcount() + 1

0    1
1    1
2    2
3    1
4    2
5    1
6    1
dtype: int64

In [3]:
df["rownum"] = df.groupby(["date", "product"]).cumcount() + 1
df

,date,product,sales,rownum
0,2023-04-01,Apple,100,1
1,2023-04-01,Banana,200,1
2,2023-04-01,Apple,150,2
3,2023-04-02,Banana,100,1
4,2023-04-02,Banana,150,2
5,2023-04-03,Apple,200,1
6,2023-04-03,Banana,180,1


In [6]:
df.sort_values(["date", "sales"], ascending=[True, False])

,date,product,sales,rownum
1,2023-04-01,Banana,200,1
2,2023-04-01,Apple,150,2
0,2023-04-01,Apple,100,1
4,2023-04-02,Banana,150,2
3,2023-04-02,Banana,100,1
5,2023-04-03,Apple,200,1
6,2023-04-03,Banana,180,1


In [7]:
df_sorted = df.sort_values(["date", "sales"], ascending=[True, False])
df_sorted

,date,product,sales,rownum
1,2023-04-01,Banana,200,1
2,2023-04-01,Apple,150,2
0,2023-04-01,Apple,100,1
4,2023-04-02,Banana,150,2
3,2023-04-02,Banana,100,1
5,2023-04-03,Apple,200,1
6,2023-04-03,Banana,180,1


In [8]:
df_sorted.groupby(["date", "product"]).cumcount() + 1

1    1
2    1
0    2
4    1
3    2
5    1
6    1
dtype: int64

In [9]:
df_sorted["sale_rank"] = df_sorted.groupby(["date", "product"]).cumcount() + 1
df_sorted

,date,product,sales,rownum,sale_rank
1,2023-04-01,Banana,200,1,1
2,2023-04-01,Apple,150,2,1
0,2023-04-01,Apple,100,1,2
4,2023-04-02,Banana,150,2,1
3,2023-04-02,Banana,100,1,2
5,2023-04-03,Apple,200,1,1
6,2023-04-03,Banana,180,1,1


In [ ]:
df_sorted[
    df_sorted["sale_rank"] == 1
][
    [
        "date",
        "product",
        "sales",
        "rownum",
    ]
]

,date,product,sales,rownum
1,2023-04-01,Banana,200,1
2,2023-04-01,Apple,150,2
4,2023-04-02,Banana,150,2
5,2023-04-03,Apple,200,1
6,2023-04-03,Banana,180,1


## ROW_NUMBER() on SQL
Show the highest price of each product on each date. All fields should be kept in the result table.
```json
{
    "date":{
        "0":"2023-04-01",
        "1":"2023-04-01",
        "2":"2023-04-01",
        "3":"2023-04-02",
        "4":"2023-04-02",
        "5":"2023-04-03",
        "6":"2023-04-03"
    },
    "product":{
        "0":"Apple",
        "1":"Banana",
        "2":"Apple",
        "3":"Banana",
        "4":"Banana",
        "5":"Apple",
        "6":"Banana"
    },
    "sales":{
        "0":100,
        "1":200,
        "2":150,
        "3":100,
        "4":150,
        "5":200,
        "6":180
    },
    "rownum":{
        "0":1,
        "1":1,
        "2":2,
        "3":1,
        "4":2,
        "5":1,
        "6":1
    }
}
```

In [17]:
df

,date,product,sales,rownum
0,2023-04-01,Apple,100,1
1,2023-04-01,Banana,200,1
2,2023-04-01,Apple,150,2
3,2023-04-02,Banana,100,1
4,2023-04-02,Banana,150,2
5,2023-04-03,Apple,200,1
6,2023-04-03,Banana,180,1


In [18]:
from pandasql import sqldf

sql = """
select
    *,
    ROW_NUMBER() over (partition by date, product order by sales desc) as sale_rank
from df
"""
sqldf(sql)


,date,product,sales,rownum,sale_rank
0,2023-04-01,Apple,150,2,1
1,2023-04-01,Apple,100,1,2
2,2023-04-01,Banana,200,1,1
3,2023-04-02,Banana,150,2,1
4,2023-04-02,Banana,100,1,2
5,2023-04-03,Apple,200,1,1
6,2023-04-03,Banana,180,1,1


In [19]:
sql = """
with sorted_sale as (
    select
        *,
        ROW_NUMBER() over (partition by date, product order by sales desc) as sale_rank
    from df
)
select
    date,
    product,
    sales,
    rownum
from sorted_sale
where sale_rank = 1
"""
sqldf(sql)

,date,product,sales,rownum
0,2023-04-01,Apple,150,2
1,2023-04-01,Banana,200,1
2,2023-04-02,Banana,150,2
3,2023-04-03,Apple,200,1
4,2023-04-03,Banana,180,1
